In [11]:
%load_ext autoreload
%autoreload 2

import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data import Document, DocumentType  # Giả định đường dẫn thư mục gốc đã được thêm vào sys.path nếu cần
from src.loader import DocumentLoader, PDFLoader, TEXTLoader, SUPPORTED_LOADERS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Tạo một thư mục tạm để chứa test data
TEST_DIR = "test_mock"
os.makedirs(TEST_DIR, exist_ok = True)
# Sample pdf
pdf_path = os.path.join(TEST_DIR, "sample.pdf")
# Tạo file không được hỗ trợ để test lỗi
unsupported_filepath = os.path.join(TEST_DIR,"sample.docx")

Đuôi file .pdf có được hỗ trợ? True
Đuôi file .docx có được hỗ trợ? False


In [18]:
print("--- TEST PDFLOADER ---")
pdfloader = PDFLoader(pdf_path)
content = pdfloader.load()
print("---")
print("Nội dung file PDF:")
print(content[:300])
print("---")
assert content is not None
print("=> Hàm PDFLoader hoạt động tốt")

--- TEST PDFLOADER ---
---
Nội dung file PDF:
AI AI VIET NAM
 All-in-One Course 2026
Project 1.2:
Xây dựng Chatbot RAG hỏi đáp tài liệu
Nguyễn Quốc Thái Nguyễn Phúc Nguyên Đinh Quang Vinh
Proofreading: Võ Thị Thanh Kiều, Lê Phương Linh Nga
I. Giới thiệu
Hãy tưởng tượng chúng ta có một file PDF dài 50 trang về một chủ đề phức tạp. Chúng ta muốn

---
=> Hàm PDFLoader hoạt động tốt


In [19]:
docloader = DocumentLoader()
print("Đuôi file .pdf có được hỗ trợ?", docloader.supports(pdf_path))
print("Đuôi file .docx có được hỗ trợ?", docloader.supports(unsupported_filepath))

assert docloader.supports(pdf_path) == True
assert docloader.supports(unsupported_filepath) == False

# Test generate ID (Đảm bảo ID là duy nhất và nhất quán cho cùng 1 file)
id1 = docloader._generate_doc_id(pdf_path)
id2 = docloader._generate_doc_id(pdf_path)
print(f"Generated ID cho file PDF: {id1}")
assert id1 == id2, "Lỗi: Hash ID không nhất quán!"

Đuôi file .pdf có được hỗ trợ? True
Đuôi file .docx có được hỗ trợ? False
Generated ID cho file PDF: 42f35231de7e30cbe6031a631de47cac516f98d738ee87f309ddb506f18d28f2


In [20]:
# 1. Test load file hợp lệ
try:
    doc = docloader.load(pdf_path)
    print("--- Thống tin Document thu được ---")
    print(f"ID: {doc.doc_id}")
    print(f"Path: {doc.file_path}")
    print(f"Type: {doc.doc_type}")
    print(f"Content:\n{doc.content}")
    
    assert doc.doc_type == DocumentType.PDF
    print("\n=> Load file thành công, metadata chính xác.")
except Exception as e:
    print(f"Lỗi khi load file hợp lệ: {e}")

print("-" * 30)

# 2. Test bắt ngoại lệ khi load file không được hỗ trợ
try:
    docloader.load(unsupported_filepath)
    print("Lỗi: Hệ thống không ném ra lỗi ValueError khi gặp file unsupported!")
except ValueError as e:
    print(f"Bắt lỗi thành công (Expected): {e}")

--- Thống tin Document thu được ---
ID: 42f35231de7e30cbe6031a631de47cac516f98d738ee87f309ddb506f18d28f2
Path: test_mock/sample.pdf
Type: DocumentType.PDF
Content:
AI AI VIET NAM
 All-in-One Course 2026
Project 1.2:
Xây dựng Chatbot RAG hỏi đáp tài liệu
Nguyễn Quốc Thái Nguyễn Phúc Nguyên Đinh Quang Vinh
Proofreading: Võ Thị Thanh Kiều, Lê Phương Linh Nga
I. Giới thiệu
Hãy tưởng tượng chúng ta có một file PDF dài 50 trang về một chủ đề phức tạp. Chúng ta muốn
tìm nhanh câu trả lời cho một câu hỏi cụ thể mà không cần đọc hết toàn bộ tài liệu. Đây chính
là bài toán mà chúng ta sẽ giải quyết trong bài viết này.
Large Language Models (LLMs)(Các mô hình ngôn ngữ lớn) là một dạng mô hình Trí tuệ
nhân tạo (AI) tiên tiến có khả năng hiểu và tạo ra văn bản giống như con người. Những ứng
dụng quen thuộc như ChatGPT hay Gemini đều hoạt động dựa trên phương pháp này. Tuy
nhiên, LLM có một hạn chế lớn: chúng chỉ biết những gì đã được huấn luyện từ trước, không
thể trả lời các thông tin về tài liệu 

In [22]:
loader = DocumentLoader()
print(f"Đang quét thư mục: {TEST_DIR}...")

documents = loader.load_directory(TEST_DIR)

print(f"Tìm thấy và xử lý thành công {len(documents)} documents.")
for i, doc in enumerate(documents):
    print(f"\n[Doc {i+1}] Path: {doc.file_path} | Type: {doc.doc_type}")
    print(f"Đoạn trích nội dung: {doc.content[:50]}...")

# Kiểm tra xem có sót file hay lẫn file .docx vào không
assert any(d.doc_type == DocumentType.MARKDOWN for d in documents), "Lỗi: Không quét được file Markdown!"

Đang quét thư mục: test_mock...
Tìm thấy và xử lý thành công 1 documents.

[Doc 1] Path: test_mock/sample.pdf | Type: DocumentType.PDF
Đoạn trích nội dung: AI AI VIET NAM
 All-in-One Course 2026
Project 1.2...


AssertionError: Lỗi: Không quét được file Markdown!